# Bayesian Optimisation with a Gaussian-Process Surrogate

## Historical problem

Bayesian optimisation uses probability not just to estimate an unknown quantity but to decide which experiment to run next. A posterior over objective functions is combined with an acquisition rule that trades off exploration and exploitation.

This notebook uses a one-dimensional expensive black-box objective so the decision process is easy to visualise.

In [ ]:
from pathlib import Path
import sys
from math import erf

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

ROOT = Path.cwd().resolve().parents[0]
SHARED = ROOT / "00_shared"
if str(SHARED) not in sys.path:
    sys.path.append(str(SHARED))

from plotting import save_fig, set_plot_style

set_plot_style()
rng = np.random.default_rng(505)

## Objective function and acquisition rule

In [ ]:
def objective(x, noise_sd=0.0):
    x = np.asarray(x)
    signal = np.sin(3.0 * x) + 0.45 * np.cos(5.0 * x) - 0.12 * (x - 2.0) ** 2
    return signal + rng.normal(0.0, noise_sd, size=np.shape(x))


def normal_pdf(z):
    return np.exp(-0.5 * z**2) / np.sqrt(2.0 * np.pi)


def normal_cdf(z):
    z = np.asarray(z)
    return 0.5 * (1.0 + np.vectorize(erf)(z / np.sqrt(2.0)))


def expected_improvement(x_grid, gp, best_y, xi=0.01):
    mean, std = gp.predict(x_grid, return_std=True)
    improvement = mean - best_y - xi
    z = np.divide(improvement, std, out=np.zeros_like(improvement), where=std > 1e-12)
    ei = improvement * normal_cdf(z) + std * normal_pdf(z)
    ei[std < 1e-12] = 0.0
    return ei, mean, std


domain = np.linspace(0.0, 4.0, 400).reshape(-1, 1)
y_true = np.sin(3.0 * domain[:, 0]) + 0.45 * np.cos(5.0 * domain[:, 0]) - 0.12 * (domain[:, 0] - 2.0) ** 2

In [ ]:
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=0.5, nu=2.5) + WhiteKernel(noise_level=0.05)

X = np.array([[0.2], [1.8], [3.7]])
y = objective(X[:, 0])
best_history_bo = [float(np.max(y))]

for _ in range(12):
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=2, random_state=0)
    gp.fit(X, y)
    ei, mean, std = expected_improvement(domain, gp, np.max(y))
    x_next = domain[np.argmax(ei)].reshape(1, 1)
    y_next = objective(x_next[:, 0])
    X = np.vstack([X, x_next])
    y = np.concatenate([y, y_next])
    best_history_bo.append(float(np.max(y)))

gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=2, random_state=0)
gp.fit(X, y)
ei, mean, std = expected_improvement(domain, gp, np.max(y))

random_histories = []
for _ in range(50):
    X_rand = rng.uniform(0.0, 4.0, size=(len(X), 1))
    y_rand = objective(X_rand[:, 0])
    random_histories.append(np.maximum.accumulate(y_rand))
best_history_rand = np.mean(np.array(random_histories), axis=0)

print(f"Best BO value: {np.max(y):.3f}")
print(f"Average final random-search value: {best_history_rand[-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 9))

axes[0].plot(domain[:, 0], y_true, color="#111111", lw=2, label="True objective")
axes[0].scatter(X[:, 0], y, color="#d62728", s=45, zorder=3, label="BO evaluations")
axes[0].plot(domain[:, 0], mean, color="#1f77b4", lw=2, label="GP posterior mean")
axes[0].fill_between(
    domain[:, 0],
    mean - 1.96 * std,
    mean + 1.96 * std,
    color="#1f77b4",
    alpha=0.2,
    label="95% interval",
)
axes[0].set_title("Surrogate posterior over the objective")
axes[0].set_xlabel("x")
axes[0].set_ylabel("f(x)")
axes[0].legend()

axes[1].plot(domain[:, 0], ei, color="#54a24b", lw=2)
axes[1].axvline(domain[np.argmax(ei), 0], color="#d62728", ls="--", label="Next choice")
axes[1].set_title("Expected improvement acquisition function")
axes[1].set_xlabel("x")
axes[1].set_ylabel("EI")
axes[1].legend()

fig.tight_layout()
save_fig(fig, Path("figs") / "bayesopt_surrogate_and_ei.png")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(1, len(best_history_bo) + 1), best_history_bo, marker="o", label="Bayesian optimisation")
ax.plot(np.arange(1, len(best_history_rand) + 1), best_history_rand, marker="o", label="Random search")
ax.set_title("Best value found so far")
ax.set_xlabel("Number of evaluations")
ax.set_ylabel("Best observed objective")
ax.legend()
fig.tight_layout()
save_fig(fig, Path("figs") / "bayesopt_vs_random.png")
plt.show()

## Interpretation

Bayesian optimisation uses the posterior twice:

- first as a surrogate for the unknown objective,
- then as a guide for where to evaluate next.

That is why it fits naturally into the chapter's `learning and decisions` branch of modern Bayesian history.

## References

- Mockus's 1970s work on Bayesian global optimisation.
- Jones, Schonlau, and Welch (1998), *Efficient Global Optimization of Expensive Black-Box Functions*.